# Causally-informed sparse risk scoring

A method for interpretable integer-coefficient risk scoring with a soft, threshold-free *causal* prior on feature selection. The TB-DLM dataset is one application; the method is designed to compose with any causal-evidence vector over features.

**Notation.** We follow Liu et al. (2022, FasterRisk): dataset $\mathcal{D} = \{(\mathbf{x}_i, y_i)\}_{i=1}^n$ with $\mathbf{x}_i \in \mathbb{R}^p$ and $y_i \in \{-1, +1\}$; coefficient vector $\mathbf{w} \in \mathbb{Z}^p$ with intercept $w_0$; sparsity $k$; coefficient bound $C$; multiplier $m > 0$; integer-rounded solution $\mathbf{w}^+$. Sample index $i \in [1, n]$, feature index $j \in [1, p]$.

## 1. Problem setup

Inputs to the FasterRisk classifier stage:

- Features $X \in \mathbb{R}^{n \times p}$, mixed binary and continuous.
- Binary target $y \in \{-1, +1\}^n$ (required by FasterRisk's logistic loss).
- Causal-prior vector $\mathbf{q} \in [0, 1]^p$, where $q_j$ encodes evidence that feature $j$ is a *cause* of the underlying signal (not merely a correlate).

The vector $\mathbf{q}$ is produced by a separate procedure (Section 3) whose target need not be binary: it can be continuous, ordinal, or multi-class as appropriate for the prior source. The TB application uses CMM on continuous $\log_2(\mathrm{MIC})$ at the prior-estimation stage; binarization happens only at the FasterRisk stage and is a constraint imposed by FasterRisk's design, not by the problem.

Output: integer-coefficient sparse logistic classifier whose support is biased toward features with high $q_j$, with bias strength controlled by a single continuous parameter $\mu \geq 0$.

## 2. Method

### 2.1 Modified objective

Standard FasterRisk (Liu et al. 2022, Eq. 1):

$$
\min_{\mathbf{w}, w_0} \; L(\mathbf{w}, w_0, \mathcal{D}) \quad \text{s.t.} \quad \|\mathbf{w}\|_0 \leq k, \; \mathbf{w} \in \mathbb{Z}^p, \; w_j \in [-C, C], \; w_0 \in \mathbb{Z},
$$

where

$$
L(\mathbf{w}, w_0, \mathcal{D}) = \sum_{i=1}^n \log\!\left(1 + \exp\!\left(-y_i (\mathbf{x}_i^\top \mathbf{w} + w_0)\right)\right).
$$

Modified objective with causal-prior bonus:

$$
\min_{\mathbf{w}, w_0} \; L(\mathbf{w}, w_0, \mathcal{D}) \;-\; \mu \sum_{j=1}^p q_j \cdot \mathbb{1}[w_j \neq 0]
$$

subject to the same constraints. Parameters: $\mu \geq 0$ (prior strength), $k$ (sparsity), $C$ (coefficient bound).

### 2.2 MAP derivation

Place a Bernoulli inclusion prior on support indicators $z_j = \mathbb{1}[w_j \neq 0]$ with $z_j \sim \text{Bernoulli}(\pi_j)$ independently across $j$, and a uniform conditional prior on $w_j \mid z_j = 1$ over $\{-C, \ldots, -1, 1, \ldots, C\}$. This is the **discrete spike-and-slab** decomposition (Mitchell & Beauchamp 1988, George & McCulloch 1993) adapted to integer coefficients. The log-prior reduces to

$$
\log p(\mathbf{w}) = \sum_{j=1}^p z_j \, \mathrm{logit}(\pi_j) + \text{const}.
$$

Setting $\pi_j = \sigma(\mu \cdot q_j)$:

$$
\log p(\mathbf{w}) = \mu \sum_{j=1}^p q_j \, \mathbb{1}[w_j \neq 0] + \text{const}.
$$

MAP estimation minimizes $L(\mathbf{w}, w_0, \mathcal{D}) - \log p(\mathbf{w})$, which is the modified objective up to a constant. The sigmoid link is the unique link function making the log-prior linear in $q_j$ with slope $\mu$ under this Bernoulli inclusion prior. Note: $q_j = 0$ gives $\pi_j = 1/2$ (uniform inclusion at zero evidence); the prior treats $q$ as evidence on a positive scale, not as a probability centered at $1/2$. Independence across $j$ is the load-bearing assumption: it gives per-feature decomposability and matches the per-feature nature of the evidence $q_j$. Collinearity-induced redundancy among features is not captured by the prior and is mitigated only by the hard sparsity cap (§8).

### 2.3 Limits

$\mu = 0$: $\pi_j = 1/2$, uniform inclusion prior, vanilla FasterRisk recovered.

$\mu \to \infty$: $\pi_j \to 1$ for any $q_j > 0$; the $k$-constrained optimum collapses to the support maximizing $\sum_{j \in S} q_j$ (top-$k$ by $\mathbf{q}$, modulo beam-search heuristics; see implementation cell §5.1).

### 2.4 Structural properties

**Linear separability across features.** The bonus decomposes as $\sum_j q_j \cdot \mathbb{1}[w_j \neq 0]$. Per-feature marginal cost is computable without recomputing global support quantities. FasterRisk's SparseBeamLR expansion (Liu et al. Alg. 2, with ExpandSuppBy1 as Alg. 4) and CollectSparseDiversePool swap (Alg. 5) remain per-feature decomposable. The same linear-in-$z_j$ structure means the bonus admits a one-line addition to RiskSlim's MIP formulation (a linear coefficient $-\mu q_j$ on the binary inclusion indicator), but we do not implement or evaluate that here (§9).

**Magnitude invariance under integer rounding** (conditional on support preservation). The bonus depends only on $\mathbb{1}[w_j \neq 0]$, not on $|w_j|$. FasterRisk's AuxiliaryLossRounding (Alg. 6) preserves support whenever the StarRaySearch multiplier $m$ is large enough that no $|m \cdot w_j|$ for $j \in \mathrm{supp}(\mathbf{w})$ rounds to zero. Under this condition, Theorem 3.1's multiplicative bound transfers to the modified objective:

$$
L(\mathbf{w}^+, w_0^+, \mathcal{D}) - L(\mathbf{w}, w_0, \mathcal{D}) \;\leq\; \sqrt{\,n \sum_{i=1}^n \sum_{j=1}^p \ell_i^2 \, x_{ij}^2 \, u_j (1 - u_j)\,},
$$

with $u_j = w_j - \lfloor w_j \rfloor$ and $\ell_i = 1 / (1 + \exp(y_i \mathbf{x}_i^\top \boldsymbol{\gamma}_i))$ as in Liu et al. The bonus term in the modified objective is identically zero across rounding whenever support is preserved. Support preservation can fail at extreme $\mu$ when the bonus forces low-$|w_j|$ features into the support (§8); Liu et al. report that FasterRisk's multiplier search empirically mitigates this on standard benchmarks, but no theoretical guarantee at extreme $\mu$.

**Scale.** $\mu$ has no data-invariant scale: $L(\mathbf{w}, w_0, \mathcal{D})$ grows with $n$ while $\mathbf{q} \in [0, 1]^p$ is unit-free. Practical convention: report $\mu$ in units of $\mathrm{median}_j \,|\nabla_j L|$ evaluated at $\mathbf{w} = \mathbf{0}$, computed once per dataset.

### 2.5 Implementation

Local modifications to FasterRisk's `sparseBeamSearch.py` (SparseBeamLR, Alg. 2) and `sparseDiversePool.py` (CollectSparseDiversePool, Alg. 5). Pure Python; no asymptotic complexity change. Implemented and tested for numerical equivalence to vanilla FasterRisk at $\mu = 0$ or $\mathbf{q} = \mathbf{0}$ on small synthetics (3 no-op configurations bit-identical to vanilla). FasterRisk's implementation also includes a small $L_2$ ridge $\lambda_2 \|\mathbf{w}\|^2$ as numerical regularization (default $\lambda_2 = 10^{-8}$, not in Liu et al. Eq. 1); we preserve this exactly. Full change-point inventory in the implementation cell.

## 3. The causal-prior constraint

**Requirement.** $\mathbf{q}$ should come from a procedure that performs conditional-independence reasoning to remove confounding-driven associations. Predictive-only signals (bootstrap stability of LASSO/RF/XGBoost, marginal mutual information, model-based feature importance) are not used as $\mathbf{q}$. Such signals are derived from the same logistic-loss objective the classifier already optimizes; treating them as a prior duplicates information rather than adding it.

The intended contribution of a causal $\mathbf{q}$ (under the assumption of faithfulness and a sound prior-source procedure) is information about which features are *causes* of the target rather than features that covary with causes through confounding paths. Without that distinction the method reduces to a sparse-classifier-with-side-information; whether causal $\mathbf{q}$ actually outperforms predictive $\mathbf{q}$ on the support-recovery axis is tested empirically on the §6.1 synthetic.

The prior-estimation procedure operates on whatever target type it accepts (continuous, binary, multi-class). Any binarization happens later, only at the FasterRisk classifier stage.

### 3.1 Admissible sources

Sources used or planned for evaluation:

1. **PC algorithm** (Spirtes et al.). Causal sufficiency assumed; fast; works on mixed types via appropriate conditional-independence tests. The default option; used as the prior source for the §6.1 generic synthetic and the §6.2 standard benchmark.
2. **GES** (Chickering). Score-based DAG learner; causal sufficiency assumed. Second source in §6.1.
3. **IAMB / HITON-MB.** Markov-blanket-targeted; appropriate when only $\mathrm{MB}(t)$ is required.
4. **Invariant Causal Prediction** (Peters et al., 2016). Pooled-environment regression with invariance-based selection; produces $p$-value-derived $q_j$ in multi-environment settings.
5. **Knowledge graphs with directional causal edges.** Posterior causal-edge probability from curated bases (e.g., gene regulatory networks with established directionality).
6. **Expert causal elicitation.** Per-feature inclusion probabilities from a domain expert, explicitly conditioned on causal status rather than predictive utility.
7. **CMM-logistic** (Mameche et al.) with FLXMRglm binomial extension. Admits latent confounders via per-node mixture components. Handles continuous and binary node types natively. Selected for the TB application (Section 7) because causal sufficiency is violated for that data.

All sources are used via subsample stability selection ($B$ runs, frequency $q_j = \mathrm{freq}(j \to t)$). Comparing source quality across these is an empirical question (Section 6). The MAP construction holds regardless of which source produces $\mathbf{q}$.

## 4. Inherited and missing guarantees

**Inherited.**

- Integer-rounding bound (Liu et al. 2022, Thm. 3.1) on $L$ transfers exactly to $L_{\text{mod}} = L - \mu \sum_j q_j \mathbb{1}[w_j \neq 0]$ under support preservation.
- For graph-based sources at subsample fraction $\leq 1/2$, Meinshausen-Bühlmann FWER bounds apply to $\mathbf{q}$. The 80% subsampling fraction used in the TB application trades this guarantee for finite-sample power.

**Not inherited.**

- No end-to-end consistency for the two-stage pipeline (prior estimation followed by MAP selection).
- No closed-form $\mu$ scaling against $n$.

## 5. Theoretical targets

Targets (none proven; planned analyses, ordered by feasibility within the implementation window):

1. **Robustness to prior perturbation.** Target: for $\mu$ above a separation threshold $\mu_0$ (depending on the loss-curvature gap between the $k$-th and $(k{+}1)$-th best supports), bound the MAP support change in $\|\mathbf{q}\|_\infty$ perturbations. Expected shape $O(\epsilon / \mu)$ from perturbation analysis; not derived.
2. **Rashomon pool as posterior region.** Target: characterize the FasterRisk Rashomon pool $\mathcal{P}$ (output of CollectSparseDiversePool) as an approximation to the posterior mode region under the Bernoulli inclusion prior, quantifying the approximation gap induced by gap-tolerance on the raw loss.
3. **(Advisor-dependent.) Consistency.** Aspirational target: under faithfulness, prior-source identifiability, and $\mathbf{q} \to \mathbb{1}[j \in \mathrm{MB}(t)]$ as $B \to \infty$, MAP recovers the true MB of the prior-source target $t$ as $n, B \to \infty$.

## 6. Evaluation framework

The method makes three distinct claims, each tested in a separate section with metrics appropriate to that claim. No single experiment validates all three; trying to bundle them is a category error this writeup made in earlier drafts.

| Claim | What it says | Where tested |
|---|---|---|
| **A. Mechanism** | The penalty $-\mu \sum_j q_j \mathbb{1}[w_j \neq 0]$ steers feature selection toward features with high $q$, monotonically in $\mu$, and behaves sensibly under noisy or adversarial $q$. | §6.1, synthetic with ground truth |
| **B. Substrate parity** | When wired to a standard benchmark, the modified FasterRisk produces a scorecard at least as good a classifier as vanilla FasterRisk. | §6.2, FICO benchmark |
| **C. Domain utility** | On the actual TB application, the method picks features consistent with biological knowledge and produces a stable, readable scorecard. | §6.3, TB-DLM data |

No results have been produced for §6.1–§6.3; §7.3 reports actual TB stability-selection results that pre-date the FasterRisk modification.

All risk-scoring experiments use FasterRisk as the solver. FasterRisk is the current SOTA for binary integer-coefficient risk scores: Liu et al. 2022 (their Figure 4) report it beats RiskSlim (Ustun & Rudin 2017) on test AUC across six standard benchmarks, attributable to its larger hypothesis class (multiplier $m > 0$ enlarges the searchable continuous-coefficient volume by a factor of $m^k$). We build on FasterRisk and do not re-litigate the FasterRisk-vs-RiskSlim comparison.

### 6.1 Mechanism test (claim A): generic causal DAG synthetic

Tests whether the prior-penalty mechanism does what it's designed to do: steer feature selection toward features with high $q$, monotonically in $\mu$, with sensible behavior under noisy or wrong $q$. **Not** a scorecard-quality test. AUC, Brier, calibration belong to claim B and are not measured here.

The setup mirrors the TB pipeline's structure: continuous $Y$ for the causal-discovery stage, binarized for the FasterRisk classifier. Planted ground truth substitutes for CMM's stability-selection output. Standard causal-discovery synthetic template (Zheng et al. 2018 NO TEARS, Peters et al. 2016 ICP, Aliferis et al. 2010): continuous features, causally sufficient DAG (no latent variables), confounding mediated only by *observed* common ancestors. This is the shared "happy case" for causal-sufficiency-assuming sources (PC, GES, IAMB); sources outside this family (ICP needs multi-environment data, CMM targets latent-confounded regimes) are out of scope here.

**Data generation.** Erdős-Rényi DAG over $p + 1$ nodes: $p$ continuous features and a continuous latent sink $Y_{\text{lat}}$. Fully linear-Gaussian SCM:

$$
Y_{\text{lat}} = \mathbf{w}^{*\top} \mathbf{x} + w_0^* + \varepsilon, \quad \varepsilon \sim \mathcal{N}(0, \sigma_{\text{noise}}^2),
$$

with planted sparse $\mathbf{w}^*$ on $S^* = \mathrm{Pa}(Y_{\text{lat}})$. Two views of $Y$ are exposed:

- $y_{\text{continuous}} = Y_{\text{lat}}$, fed to the causal-discovery stage (PC, GES). Fisher Z applies throughout the DAG; no mixed-data CI tests needed.
- $y = 2 \cdot \mathbb{1}[Y_{\text{lat}} > \tau] - 1$, fed to FasterRisk. Default $\tau = \mathrm{median}(Y_{\text{lat}})$ for ~50/50 class balance.

This is structurally the TB pipeline applied to synthetic data, with the additional benefit that ground truth $S^*$ is known. Implementation: `src/data/synthetic_lingauss.py`.

**Why this regime, and honest acknowledgment.** §6.1 is the mechanism check, so we isolate the question "does the prior penalty act on $\mathbf{q}$ correctly?" from the orthogonal question "does PC handle mixed binary/continuous data well?". By giving PC clean Fisher Z conditions on $y_{\text{continuous}}$, $\mathbf{q}$ is as good as PC can produce; the mechanism is then tested on a high-quality input, which is the cleanest condition under which a positive result is meaningful (and a negative result is informative). The "what if $\mathbf{q}$ is noisier" question is probed separately by the $\sigma$-sweep on $\mathbf{q}$.

This is **deliberately favorable to PC**. We do not claim §6.1 demonstrates that causal $\mathbf{q}$ beats predictive $\mathbf{q}$ in general; it demonstrates that the mechanism behaves as designed under controlled conditions. The §6.2 FICO benchmark (natively binary $Y$, mixed-data CI tests, no ground truth) covers the realistic-deployment regime where PC's CI tests are weaker; §6.3 (TB) covers the real application.

**Prior sources compared.**

1. **Oracle.** $q_j = 1$ for $j \in S^*$, $0$ elsewhere, plus Gaussian noise $\sigma$ clipped to $[0,1]$. Upper bound on what a perfect causal procedure could provide.
2. **PC stability.** PC on $(X, y_{\text{continuous}})$ with Fisher Z, $B = 100$ subsampling at fraction $1/2$ (preserves M-B FWER control); $q_j = \mathrm{freq}(j \to Y)$. The realistic constraint-based source.
3. **GES stability.** Greedy Equivalence Search (causal-learn) on $(X, y_{\text{continuous}})$ with Gaussian BIC, $B = 100$ subsampling at fraction $1/2$; $q_j = \mathrm{freq}(j \to Y)$. Second sufficiency-assuming source; checks whether the mechanism's behavior is PC-specific.

   *Implementation note.* causal-learn and pgmpy GES are pure-Python with weak score caching; on dense Gaussian DAGs at $p \geq 20$ a single subsample call can exceed 30 minutes wall time. `ges_stability_q` (in `src/causal_prior/priors.py`) caps each subsample at a 1800s SIGALRM timeout; timed-out subsamples contribute zeros to the q estimate, and `ges_n_timeouts` is recorded per cell. At $p = 30, p_{\text{edge}} \geq 0.4$ a non-trivial fraction of subsamples saturates the timeout, so $q_{\text{ges}}$ is conservative on those cells. R's pcalg::ges (in C) finishes in seconds on the same shapes but is outside the Python stack; switching backends is left as future work.
4. **Bootstrap-$L_1$.** $L_1$-logistic regression on $(X, y)$, $B = 100$ subsamples; $q_j = \mathrm{freq}(\hat\beta_j \neq 0)$. The predictive baseline. Bootstrap-$L_1$ peaks on confounded correlates of $Y$; PC and GES peak on $S^*$. This is the causal-vs-predictive comparison made operational.
5. **Uniform.** $q_j$ constant. Tests whether prior *structure* (not just per-feature mass shift) carries the signal.
6. **Adversarial.** $q_j = 1$ on the confounded-correlate set $C$ (non-$S^*$ features d-connected to $Y$), $0$ elsewhere. Failure-mode probe: at small $\mu$ the loss term should dominate and recover vanilla; at large $\mu$ the prior should override and the support should drift onto the wrong features. Informative either way.

**Sweep.** The mechanism claim has three sub-axes; each needs its own sweep.

- **Headline: confounding × prior strength.** Fix $(p, n, k_\star, K) = (30, 300, 5, 10)$, where $K = 2 k_\star$ is FasterRisk's sparsity budget (see *K choice* below). Sweep $p_{\text{edge}} \in \{0.1, 0.2, 0.3, 0.4, 0.5, 0.6\}$ (low to high confounding) and $\mu$ over a 13-point grid (the $\mu = 0$ vanilla anchor plus 12 log-spaced points in $\mu_{\text{relative}} \in [10^{-2}, 10^1]$, with $\mu_{\text{relative}} = \mu / \mu_{\text{scale}}$) across all six prior sources, with 5 seeds per cell. This is the load-bearing experiment; it tests whether the causal-vs-predictive gap exists in the regime where the prior has support-level headroom, i.e., where vanilla loses S* recovery. The extended $p_{\text{edge}}$ sweep is needed because at low $p_{\text{edge}}$ vanilla saturates at recall $= 1.0$ and the prior cannot move the support; the gap is only observable above a $p_{\text{edge}}$ threshold where confounders crowd S* out of the K-feature support.
- **Oracle noise robustness.** At the anchor cell $(p_{\text{edge}} = 0.2$, $\mu = \mu_{\text{mid}})$ of the headline sweep sweep Gaussian noise on the oracle $\mathbf{q}$ at $\sigma \in \{0, 0.1, 0.3, 0.5\}$. Small, separate experiment with 5 seeds; produces the graceful-degradation curve. $\sigma$ is applied only to the oracle source because PC, GES, and bootstrap-$L_1$ are already noisy by construction.
- **Op-point robustness** (reviewer-answer): one or two additional $(p, n, k_\star)$ settings, e.g., $(20, 300, 3)$ and $(50, 1000, 7)$, run only on the $\mu$-monotonicity axis at $p_{\text{edge}} = 0.2$. Addresses "is this an artifact of one configuration?" without rerunning the full grid.

$\mu$ has no data-invariant scale (§2.4); $\mu_{\text{scale}} = \mathrm{median}_j \,|\nabla_j L|$ at $\mathbf{w} = \mathbf{0}$ is computed once per dataset and $\mu_{\text{relative}} = \mu / \mu_{\text{scale}}$ is the unit used in cross-cell plots.

**K choice.** At $K = k_\star$ FasterRisk saturates at full S* recovery whenever the underlying signal is separable, leaving no support-level slot for the prior to influence; the only observable would be tie-breaking among ties. $K = 2 k_\star$ gives FR $K - k_\star = k_\star$ extra slots to fill, which vanilla allocates to the strongest non-causal predictors (confounded correlates of Y). The prior's effect lives in how those extra slots get used: a $q$ concentrated on S* (oracle, PC when discovery has enough signal) leaves the slots unfilled or shrinks the support; a $q$ that also marks confounders (bootstrap-$L_1$) confirms FR's existing choices; an adversarial $q$ pulls slots onto C at the cost of dropping S* features. The prior's effect is observable in precision throughout, and in recall once vanilla fails at high $p_{\text{edge}}$.

**Metrics (mechanism only).**

- **Support recovery vs $S^*$.** Overlap $|\hat S \cap S^*| / k$, precision $|\hat S \cap S^*| / |\hat S|$, recall $|\hat S \cap S^*| / |S^*|$. Headline metric, reported per (source, $\mu$, $p_{\text{edge}}$) cell.
- **Monotonicity in $\mu$.** For sound sources (oracle, PC, GES), recovery should increase monotonically in $\mu$; for adversarial $\mathbf{q}$, it should *decrease* monotonically.
- **Selectivity-driven recovery, where headroom exists.** Define the *selectivity ratio* of a $q$ source as $\mathrm{sel}(q) = \bar q_C / \bar q_{S^*}$, where $\bar q_C = |C|^{-1} \sum_{j \in C} q_j$ and $\bar q_{S^*} = |S^*|^{-1} \sum_{j \in S^*} q_j$; lower means more causally-selective. Headline plot: $\mathrm{recovery}(q)$ at fixed $\mu_{\text{relative}}$ in the headroom region, plotted against $\mathrm{sel}(q)$ across the six sources. The prediction: support recovery is a monotone-decreasing function of $\mathrm{sel}(q)$, regardless of whether $q$ came from a causal or predictive procedure. Under linear-Gaussian conditions on the cached cells GES produces $\mathrm{sel}(q_{\text{GES}}) \approx 0.03$ at $p_{\text{edge}} = 0.1$, bootstrap-$L_1$ produces $\mathrm{sel}(q_{L_1}) \approx 0.43$, and PC sits between (0.02 at $p_{\text{edge}} = 0.1$ but rapidly losing $\bar q_{S^*}$ with confounding, so $\mathrm{sel}$ degenerates). This is a stronger statement than "causal beats predictive": within the causal family PC and GES differ by an order of magnitude in retained selectivity as $p_{\text{edge}}$ grows (constraint-based discovery is more sample-starved than score-based on Gaussian DAGs), so the realized headline effect tracks selectivity, not provenance.
- **Oracle noise robustness.** Recovery as a function of $\sigma$ on the oracle $\mathbf{q}$, at the anchor cell of the noise sweep.
- **Soft-vs-hard prior comparison.** For each causal or predictive source $\mathbf{q} \in \{q_{\text{PC}}, q_{\text{GES}}, q_{L_1}\}$, recovery under the soft prior at a fixed $\mu_{\text{relative}}$ in the headroom region vs the hard pre-selection baseline $\hat{S}_t = \{j : q_j \geq t\}$ + vanilla FasterRisk on the restricted columns at $K' = \min(K, |\hat{S}_t|)$, swept over $t \in \{0.3, 0.5, 0.7\}$ (Meinshausen-Bühlmann conventional thresholds). Tests whether propagating q-uncertainty through the optimization (soft) outperforms binarizing it upfront (hard). Detailed baseline methodology in §6.4.
- **Stability across CV folds.** Jaccard overlap of $\hat S$ across 5-fold CV. Secondary.

**What §6.1 demonstrates if it works.** The mechanism steers selection toward high-$\mathbf{q}$ features in proportion to $q$'s selectivity ($\mathrm{sel}(q) = \bar q_C / \bar q_{S^*}$). Sources with low $\mathrm{sel}$ (oracle, GES on Gaussian DAGs) drive recovery toward $S^*$; sources with high $\mathrm{sel}$ (bootstrap-$L_1$, which marks confounders alongside causes) drive recovery toward whichever features carry strong predictive signal, including confounders. Causal provenance is necessary but not sufficient: PC and GES are both causal by construction, but PC's $\bar q_{S^*}$ collapses sharply with confounding (constraint-based discovery is sample-starved on dense Gaussian DAGs) while GES retains selectivity into the moderate-confounding regime; the realized headline effect tracks selectivity within the causal family, not provenance across families. Uniform $\mathbf{q}$ behaves like vanilla (no per-feature differential, no mechanism input). Adversarial $\mathbf{q}$ produces a controlled failure as $\mu$ grows; the mechanism is responsive to the prior, not robust to it being wrong.

**What §6.1 explicitly does not show.** That causal $\mathbf{q}$ beats predictive $\mathbf{q}$ in general; the observed effect tracks selectivity, not provenance, and the usable $p_{\text{edge}}$ regime is bounded above by whatever density still leaves at least one discovery method with usable selectivity. Beyond that boundary all sources are sample-starved and the prior has no useful signal to consume (a property of the upstream discovery step, not of the soft-penalty mechanism). The regime is deliberately favorable to score-based and constraint-based discovery (Gaussian DAG, continuous $Y$, Fisher Z); the §6.2 FICO benchmark and §6.3 TB application cover the realistic-deployment regime where these assumptions break.

### 6.2 Substrate parity (claim B): FICO benchmark

Tests that the modified FasterRisk produces a scorecard at least as good a classifier as vanilla FasterRisk on a published benchmark. No causal ground truth exists on a real benchmark, so **no support-recovery claim is made here**. The benchmark serves a sanity-check role: does wiring the prior break the FasterRisk substrate?

**Dataset.** FICO Home Equity Line of Credit (Explainable ML Challenge), $n \approx 10{,}000$, 23 original features (mostly continuous credit scores, durations, ratios). Used in Liu et al. 2022 and Ustun & Rudin 2017. Chosen over mammo because mammo's five original features give PC almost nothing to discover; the standard binarization then expands to mostly one-hot binary columns, where constraint-based causal discovery has known weaknesses. FICO has substantial continuous content that PC can exploit.

**Two-stage protocol.**

1. **Causal discovery on the original (pre-binarized) features.** Run PC with subsampling stability ($B = 100$, fraction $1/2$) on the 23 original FICO features, using a CI test appropriate for the mixed binary/continuous content (pillai or partial correlation). Output: $q^{\mathrm{orig}}_j \in [0, 1]$ for each original feature.
2. **Map $q^{\mathrm{orig}}$ to the binarized columns.** For each binarized column $c$ (e.g., `ExternalRiskEstimate_lt_70`), set $q^{\mathrm{bin}}_c = q^{\mathrm{orig}}_{\mathrm{parent}(c)}$, where $\mathrm{parent}(c)$ is the original feature $c$ was derived from. Justification: the causal structure lives at the original-feature level. Binarized columns are downstream encoding choices and inherit the causal status of their parent feature.

**Comparison.** Vanilla FasterRisk ($\mu = 0$) vs causal FasterRisk ($\mu > 0$, sweep over 3–5 values), at matched sparsity $k$.

**Metrics (predictive only).**

- **Test AUC** on a held-out split (mean ± std across $S = 10$ random splits).
- **Brier score** and **ECE** for calibration.
- **Number of selected features**, to verify sparsity matches.

**What §6.2 demonstrates if it works.** Causal FasterRisk's AUC matches (or beats) vanilla FasterRisk's at matched sparsity, on a standard benchmark. The modification doesn't degrade the FasterRisk substrate. Necessary but not sufficient for the method to be useful; the causal-correctness question lives on the synthetic, the domain-utility question lives on TB.

### 6.3 Domain utility (claim C): TB-DLM case study

Tests whether the method, applied to the actual TB application, picks features consistent with biological knowledge and produces a stable, readable scorecard. Three sub-experiments differ in feature pre-filtering ($n = 164$ throughout). The prevalence filter $[5\%, 98\%]$ is a data-quality gate (drops near-constant features); it applies to all arms identically and is not a feature-selection method.

**Experiment 3.1. Raw mutation set ($p = 211$).** Both arms operate on the full mutation matrix. The causal-prior vector has nonzero entries only on the 17 prevalence-feasible features that CMM could evaluate; $q_j = 0$ on the rest encodes "no causal evidence available" (uniform contribution at $\mu = 0$). The $p \gg n$ regime is where we hypothesize the causal prior would most reduce overfitting; hypothesis, not result.

- Vanilla FasterRisk ($\mu = 0$) at $p = 211$
- Causal FasterRisk ($\mu > 0$) at $p = 211$

**Experiment 3.2. Prevalence-filtered set ($p = 17$).** Vanilla vs causal FasterRisk at matched $(n, p, k)$.

**Experiment 3.3. Post-causal-selection scorecard.** Take the survivors from §7.3 (`pepq_Ala87Gly` + lineage indicators + any second-pass candidates), fit a FasterRisk scorecard at $k = 3$. Exhibit, not comparison: shows that causal pre-selection shrinks the problem to a scorecard small enough to read off by hand.

**Metrics (domain utility, not classification accuracy).**

- **Support stability across CV folds** (Jaccard overlap, 10-fold CV). Headline metric. At $n = 164$ with small effect sizes, *which features each method picks across folds* is a better discriminator than test AUC, whose confidence intervals will overlap between methods.
- **Qualitative reading of selected features.** Are the picks biologically plausible? Do they match the §7.3 stability-selection candidates? Do they line up with the (limited) prior literature on DLM resistance?
- **Scorecard readability.** Final exhibit: is the scorecard small enough for a clinician to evaluate at a glance?

**What §6.3 does not measure as the headline.** Test AUC as a standalone claim. At $n = 164$ the confidence intervals are too wide for the predictive comparison to be the headline result; stability and biological plausibility are the more informative axes at this sample size.

### 6.4 Baselines and ablations

In-scope across §6.1–§6.3:

- **Vanilla FasterRisk** ($\mu = 0$). Always present in §6.1, §6.2, §6.3.
- **Hard pre-selection** ($\mu \to \infty$). Tests the §2.3 upper limit (§6.1 only).
- **Uniform $\mathbf{q}$ ablation.** Same $\mu$, $q_j$ constant; tests whether prior *structure* carries the signal (§6.1).
- **Bootstrap-$L_1$ as $\mathbf{q}$.** Predictive baseline; the causal-vs-predictive ablation. **§6.1 only.** Running bootstrap-$L_1$ alongside the CMM stability pipeline on TB is too expensive for the internship window.
- **Adversarial $\mathbf{q}$.** Failure-mode probe (§6.1 only).

Out-of-scope (future work):

- $L_1$ logistic regression, XGBoost (capacity upper bound), modern causal feature selection (2023–2025 methods). Standard external baselines; can be added at writeup time if room.
- **RiskSlim port of the prior.** The penalty admits a one-line addition to RiskSlim's MIP formulation (linear coefficient on the inclusion indicator). Cross-solver evaluation would test whether the prior's gain depends on hypothesis class, but FasterRisk is the SOTA solver and the internship focuses on within-solver causal-vs-vanilla and causal-vs-predictive comparisons.

### 6.5 Honest concerns

- **$n = 164$ on TB is small.** All CV-derived metrics on §6.3 will carry wide intervals. Support stability is the headline for that reason.
- **$q_j = 0$ encodes "no causal evidence available," not "known non-causal".** At $\mu = 0$ this contributes uniformly; at $\mu > 0$ the no-evidence features are effectively deprioritized. Matters in §6.3 Exp 3.1, where most features sit at $q_j = 0$ by construction.
- **Single-solver evaluation.** All experiments use FasterRisk. Whether the prior's gain transports to RiskSlim's MIP formulation (smaller hypothesis class, no multiplier) is open and is future work.
- **§6.2 q-propagation step is novel-ish.** "Inherit $q$ from parent feature" is the natural move given the two-stage architecture, but I'm not aware of a paper that formalizes it for this purpose. Needs careful framing in the writeup; presented as a deliberate methodological choice with reasoning, not as a free move.

## 7. Application: TB-DLM resistance

### 7.1 Data and targets

$n = 164$ TB-MDR strains tested against delamanid; $p \approx 14$ mutations after prevalence filtering at $[5\%, 98\%]$; 3 lineage indicators with lineages 1 ($n = 3$) and 3 ($n = 4$) merged into reference; `type_beyond_MDR` (clinical preXDR/XDR vs. MDR) as an optional sensitivity covariate.

**The natural target is continuous.** $t = \log_2(\mathrm{MIC})$ on a 2-fold dilution series. The CMM prior-estimation stage operates directly on $t$: MIC is a sink node in the DAG with a linear-Gaussian mechanism, while the binary mutation nodes use the logistic mixture mechanism (FLXMRglm).

**Binarization is required by FasterRisk, not by the problem.** FasterRisk requires $y \in \{-1, +1\}^n$, so we obtain it as $y = 2 \cdot \mathbb{1}[\mathrm{MIC} > \tau] - 1$. Three thresholds compared (median split, top quartile, agar-based `interp_dlm016`); binarization sensitivity is part of the evaluation.

This two-stage continuous-then-binary architecture is application-specific. Applications whose natural target is already binary (e.g., the §6.2 FICO benchmark) collapse this to one stage.

### 7.2 Prior source: CMM-logistic with stability selection

CMM (Mameche et al.) extends score-based Bayesian-network learning with per-node latent mixture components. The logistic extension (FLXMRglm, binomial link) handles binary mutation predictors directly. Continuous nodes (MIC) use the linear-Gaussian mechanism. Latent components proxy for unobserved patient-level confounders (treatment history, host factors).

**Argument for CMM on this dataset.** (i) Latent confounders are real and unobserved at this dataset's collection time; PC/GES assume causal sufficiency. (ii) Features are predominantly binary with low-prevalence tails, ill-suited to PC's standard CI tests on continuous variables. (iii) $n = 164$ is too small for heavier latent-variable structure-learners (LMB-CSEM, EM over latent subspaces). (iv) CMM handles a continuous target node natively, which matches MIC's data type. This argument is local to TB-shaped data; it does not claim CMM is the only possible choice, just the one we use here.

**Empirical source validation on TB-shaped synthetic.** The source-choice argument above is supported by two in-house synthetic experiments on mixed binary/continuous data matching the TB regime (binary low-prevalence features, continuous target node, latent mixing components):

- `notebooks/mixed_cmm/synthetic/baselines_sweep_plots.ipynb`: Figure D.2-style 4-method comparison (CMM-logistic vs PC vs GES vs empty graph) at default config ($n_\text{obs}=10$, $p_\text{graph}=0.4$, $p_\text{mix}=0.5$, $n_\text{mix}=2$, $n_\text{samples}=164$, matched to the TB application). Overall and edge-type-split (bin→bin, bin→cont) F1/precision/recall.
- `notebooks/mixed_cmm/synthetic/sweep_plots.ipynb`: Gaussian-vs-Logistic CMM driver ablation, validating the FLXMRglm binomial extension on binary mutation nodes.

These validate the prior-source choice. The full integration (FasterRisk-with-causal-prior on TB-shaped data) is exercised for the first time on the real-TB experiments in §6.3.

**Forbidden edges.** $\mathrm{MIC} \to \mathrm{mutation}$ (temporal/mechanistic); edges into lineage (lineage fixed at infection); edges into `type_beyond_MDR` (set independently of DLM). Optionally $\mathrm{lineage} \to \mathrm{MIC}$ forbidden when treating lineage strictly as a covariate. Under these constraints MIC is a DAG sink, so $\mathrm{MB}(\mathrm{MIC}) = \mathrm{Pa}(\mathrm{MIC})$ (Aliferis et al. 2010).

**Stability selection on TB data.** $B = 100$ runs at 80% subsampling. Per-run column filter at $\mathrm{mcc} \times k_{\max} = 24$ positives (below this, $k$-component logistic mixtures collapse on the binary mutation nodes). Eligibility-weighted per-edge frequencies. The 80% subsampling fraction trades the M-B familywise-error guarantee for finite-sample power at $n = 164$.

### 7.3 Findings on the real TB-DLM dataset

Four CMM stability-selection ablations completed: baseline, +lineage, +lineage+type, +lineage with $\mathrm{type} \to \mathrm{MIC}$ forbidden. Decision rule applied for further analysis: variant retained as a candidate if stable in both `+lineage` and the type-forbid spec.

- `pepq_Ala87Gly`: stable in both (0.51 in `+lineage`, 0.61 in the type-forbid spec). The only candidate that passes the cross-spec stability rule.
- `mmpl5_Asp767Asn`: appears in the type-forbid spec only; lineage_2 collinearity ($\rho = 0.95$) is a plausible explanation but not formally tested.
- `rv1979c_C-135G`: monotonically degrades across specs; does not pass the rule.

Caveats: $n = 164$ from a single geographic/clinical cohort. The stability-selection statistic is informative about identifiability within this cohort, not about transportability. Any clinical claim about `pepq_Ala87Gly` would require replication on an independent cohort.

## 8. Scope and limitations

- **Prior source selection is application-dependent.** Choice depends on assumed structure (causal sufficiency or not) and data shape (binary/continuous, $n$, target type). No general best.
- **$\mu$ tuning is empirical.** No closed-form scaling against $n$.
- **CMM is justified for TB-shaped data**; the argument in §7.2 does not extend to general settings.
- **Faithfulness** is required wherever a causal graph is the prior source. Failures show up as biased $\mathbf{q}$; the planned synthetic robustness analysis (§6.1) is intended to characterize this but has not been run.
- **Magnitude-vs-support tension at large $\mu$.** The bonus can drive features into the support with small $|w_j|$ that AuxiliaryLossRounding may eliminate when the StarRaySearch multiplier $m$ is too small.
- **Binarization is lossy on TB.** Continuous MIC carries sub-threshold variation that the binary call discards.
- **Independence of $z_j$ across $j$ ignores feature redundancy.** Under strong collinearity (TB's lineage_2 / `mmpl5_Asp767Asn` at $\rho = 0.95$), per-feature $q_j$ does not capture the "select one, not both" structure of the optimal support. The hard sparsity cap mitigates this only when $k$ is tight relative to the redundancy.
- **Single-solver evaluation.** All experiments use FasterRisk. Whether the prior's gain transports to RiskSlim's MIP formulation (smaller hypothesis class, no multiplier) is open and is future work.
- **Three separate validation contexts, no single experiment combines them.** Claim A (mechanism) is tested on synthetic where ground truth is available. Claim B (substrate parity) is tested on a real benchmark where no causal ground truth exists. Claim C (domain utility) is tested on TB where domain knowledge substitutes for ground truth. This separation is honest: no single setting can test all three simultaneously, and trying to bundle them is the kind of category error this writeup tries to avoid.

## 9. Methodological contributions

These are the intended contributions; the empirical evidence is planned (§6), not produced.

1. **Bayesian causal-prior penalty for sparse integer classification.** Bernoulli inclusion prior with sigmoid link yields a linear-in-$\mathbf{q}$ MAP bonus. Threshold-free one-parameter family with vanilla FasterRisk and hard pre-selection as the two limits. The linear-in-$q$ form follows from the Bernoulli + sigmoid construction; alternative link functions yield non-decomposable log-priors.
2. **Decomposability-preserving integration into FasterRisk.** Linear separability of the bonus preserves SparseBeamLR and CollectSparseDiversePool per-feature decomposability; AuxiliaryLossRounding bound (Thm. 3.1) transfers under support preservation. Implemented; numerical equivalence to vanilla at $\mu = 0$ verified.
3. **A general interface for causal evidence in sparse integer classifiers.** The method composes with any procedure outputting $\mathbf{q} \in [0, 1]^p$ (§3.1). The prior-estimation target need not be binary; only the FasterRisk classifier stage requires binary input. The causal-vs-predictive question is tested on synthetic in the generic regime (§6.1, PC and GES sources). Extension to RiskSlim's MIP formulation is structurally straightforward (the penalty is a linear coefficient on the binary inclusion indicator) and is left to future work.
4. **Logistic extension of CMM** (FLXMRglm binomial link integrated into the mixture-EM structure-learning loop), as the prior source for the TB application. The defensibility of this choice is local to TB-shaped data (§7.2), supported by the in-house CMM-vs-PC-vs-GES and Gaussian-vs-Logistic baselines sweeps on mixed binary/continuous synthetic data.

# Implementation plan: FasterRisk modification

Reference for the FasterRisk code changes implementing the modified objective from cell 0, §2.1. The implementation is source-agnostic: it does not depend on how $\mathbf{q}$ is produced, only on its values.

**Notation (math ↔ code).** Following Liu et al. (2022):

| Math | Code | Meaning |
|---|---|---|
| $\mathbf{w} \in \mathbb{Z}^p$ | `self.betas` | coefficient vector |
| $w_0$ | `self.beta0` / intercept | intercept |
| $\mathbf{w}^+$ | rounded `betas` | integer-rounded coefficients |
| $L(\mathbf{w}, w_0, \mathcal{D})$ | `compute_logisticLoss_*(...)` | logistic loss |
| $k$ | `k` (or sparsity arg) | sparsity bound |
| $C$ | `original_lb` / `original_ub` | coefficient box bound |
| $\lambda_2$ | `self.lambda2` | $L_2$ ridge (default $10^{-8}$) |
| $m$ | StarRaySearch multiplier | multiplier $m > 0$ |
| $\mathbf{q} \in [0,1]^p$ | `self.freq` | causal-prior vector (new) |
| $\mu \geq 0$ | `self.mu` | prior strength (new) |

The numpy array `self.freq` has shape $(p,)$ indexed consistently with feature columns. The name `freq` reflects its origin as stability frequencies on a causal graph, but the implementation does not require this.

**Backward-compatibility guarantee.** At $\mu = 0$ or `freq = zeros(p)`, the modified code is numerically equivalent to vanilla FasterRisk.

## 1. FasterRisk in three stages

(Algorithm 1 in Liu et al. 2022.)

1. **SparseBeamLR** (Alg. 2): beam search for a sparse continuous coefficient vector $(\mathbf{w}^*, w_0^*)$ with $\|\mathbf{w}^*\|_0 \leq k$, $\|\mathbf{w}^*\|_\infty \leq C$. Calls `ExpandSuppBy1` (Alg. 4) per beam level. Selection by smallest logistic loss at fixed support size.
2. **CollectSparseDiversePool** (Alg. 5): given $(\mathbf{w}^*, w_0^*)$, build a Rashomon pool $\mathcal{P}$ of near-optimal alternatives by single-feature swaps. Accept a swap if its loss is within $(1 + \epsilon) L^*$.
3. **StarRaySearch** (Alg. 3) + **AuxiliaryLossRounding** (Alg. 6): for each pool member, search over multipliers $m > 0$ and round to integers $\mathbf{w}^+$. Multiplicative loss-error bound (Thm. 3.1).

The modification targets stages 1 and 2 (support selection). Stage 3 is untouched because it operates on fixed supports; the bonus is invariant under any operation preserving support.

## 2. Change-point inventory

### 2.1 Files

All under `external/fasterrisk/src/fasterrisk/`:

| File | Modification |
|---|---|
| `sparseBeamSearch.py` | 2 sites (CP1, CP2) |
| `sparseDiversePool.py` | 3 sites (CP3, CP4, CP5) |
| `fasterrisk.py` | constructor threading (CP6) |
| `wrapper.py` | constructor threading (CP6) |
| `base_model.py`, `rounding.py`, `utils.py` | no modification |

Pure Python. No Cython, no C++.

### 2.2 CP1: gradient ranking in beam-search expansion

**File:** `sparseBeamSearch.py`, inside `expand_parent_i_support_via_OMP_by_1`, around lines 42-46.

**Original:**
```python
grad_on_non_support = self.yXT[non_support].dot(np.reciprocal(1 + self.ExpyXB_arr_parent[i]))
abs_grad_on_non_support = np.abs(grad_on_non_support)
num_new_js = min(child_size, len(non_support))
new_js = non_support[np.argsort(-abs_grad_on_non_support)][:num_new_js]
```

Picks the top `child_size` non-support features to expand into. Ranking is by $|\nabla_j L|$ at coefficient zero (first-order Taylor proxy for loss reduction).

**Modified:**
```python
grad_on_non_support = self.yXT[non_support].dot(np.reciprocal(1 + self.ExpyXB_arr_parent[i]))
abs_grad_on_non_support = np.abs(grad_on_non_support) + self.mu * self.freq[non_support]
num_new_js = min(child_size, len(non_support))
new_js = non_support[np.argsort(-abs_grad_on_non_support)][:num_new_js]
```

**Justification.** The first-order marginal change in $L_{\text{mod}}$ when adding feature $j$ is approximately $-|\nabla_j L| - \mu \cdot q_j$. Ranking by $|\nabla L| + \mu \cdot \mathbf{q}$ in descending order ranks by first-order modified-objective reduction.

**Limit checks.** $\mu = 0$: identical to vanilla. $\mu \to \infty$: pure top-$k$ by $\mathbf{q}$.

Strictly, the optimal first-order reduction of $L_{\mathrm{mod}}$ on adding feature $j$ is $|\nabla_j L|^2/(2H_{jj}) + \mu q_j$; under FasterRisk's standardization $H_{jj}$ is approximately constant across $j$, so ranking by $|\nabla_j L| + \mu q_j$ is rank-equivalent to ranking by the optimal reduction up to the same approximation vanilla FasterRisk uses. CP2 corrects to exact $L_{\mathrm{mod}}$ at the beam-survivor step.


### 2.3 CP2: child-loss in beam-search keep/discard

**File:** `sparseBeamSearch.py`, line 82.

**Original:**
```python
self.loss_arr_child[child_id] = compute_logisticLoss_from_ExpyXB(self.ExpyXB_arr_child[child_id])
```

Stored child loss is used at line 100 in `np.argsort(self.loss_arr_child)` to select the top `parent_size` children for the next beam level. This is where the actual beam selection happens.

**Modified:**
```python
support_idx = get_support_indices(self.betas_arr_child[child_id])
self.loss_arr_child[child_id] = (
    compute_logisticLoss_from_ExpyXB(self.ExpyXB_arr_child[child_id])
    - self.mu * self.freq[support_idx].sum()
)
```

**Justification.** Without this, CP1 biases candidate *generation* but the beam *survivor* selection still uses raw $L$, inconsistent with the modified objective.

**$L_2$ ridge asymmetry note.** Vanilla FasterRisk stores raw $L$ here (no $\lambda_2 \|\mathbf{w}\|^2$), while the diverse pool (CP4) stores $L + \lambda_2 \|\mathbf{w}\|^2$. We preserve this vanilla quirk: subtract bonus from raw $L$ in CP2, from $L + \lambda_2 \|\mathbf{w}\|^2$ in CP4. Required for numerical equivalence to vanilla at $\mu = 0$.

### 2.4 CP3: gradient ranking in diverse-pool swap

**File:** `sparseDiversePool.py`, inside `get_sparseDiversePool`, around lines 84-88.

**Original:**
```python
grad_on_availableIndices = -self.yXT[availableIndices].dot(np.reciprocal(1 + sparseDiversePool_ExpyXB[sparseDiversePool_start]))
abs_grad_on_availableIndices = np.abs(grad_on_availableIndices)
new_js = availableIndices[np.argsort(-abs_grad_on_availableIndices)[:max_num_new_js]]
```

**Modified:**
```python
grad_on_availableIndices = -self.yXT[availableIndices].dot(np.reciprocal(1 + sparseDiversePool_ExpyXB[sparseDiversePool_start]))
abs_grad_on_availableIndices = np.abs(grad_on_availableIndices) + self.mu * self.freq[availableIndices]
new_js = availableIndices[np.argsort(-abs_grad_on_availableIndices)[:max_num_new_js]]
```

Structurally identical to CP1. Swap-candidate selection biased toward features with high causal evidence.

### 2.5 CP4: pool-loss for final argsort

**File:** `sparseDiversePool.py`, lines 66 (seed-solution loss) and 102 (accepted-swap loss).

**Modified line 66:**
```python
sparseDiversePool_loss[-1] = (
    compute_logisticLoss_from_ExpyXB(self.ExpyXB)
    + self.lambda2 * self.betas[nonzero_indices].dot(self.betas[nonzero_indices])
    - self.mu * self.freq[nonzero_indices].sum()
)
```

**Modified line 102:**
```python
swap_support = np.setdiff1d(np.append(nonzero_indices, new_j), [old_j])
sparseDiversePool_loss[sparseDiversePool_index] = (
    compute_logisticLoss_from_ExpyXB(sparseDiversePool_ExpyXB[sparseDiversePool_index])
    + self.lambda2 * (betas_no_old_j_squareSum + sparseDiversePool_betas[sparseDiversePool_index, new_j] ** 2)
    - self.mu * self.freq[swap_support].sum()
)
```

**Justification.** At line 104, `np.argsort(sparseDiversePool_loss)[:totalNum_in_diverseSet][:select_top_m]` selects the final pool. Subtracting the bonus makes the pool surface near-optimal solutions under the modified objective.

### 2.6 CP5 (deliberate non-modification): gap-tolerance check stays on raw loss

**File:** `sparseDiversePool.py`, line 95.

**Why this is not modified.** The gap check is multiplicative-relative: $(L_{\text{swap}} - L_{\text{ref}}) / L_{\text{ref}} < \text{gap\_tolerance}$. At large $\mu$, $L_{\text{mod}}$ can go negative ($\mu \sum_j q_j$ exceeds $L + \lambda_2 \|\mathbf{w}\|^2$); dividing by a negative reference flips the inequality and the gap check accepts arbitrarily bad swaps. Keep the gap comparison on raw $L + \lambda_2 \|\mathbf{w}\|^2$ (always nonnegative).

**Semantic consequence.** Pool members are "near-optimal in raw $L + \lambda_2 \|\mathbf{w}\|^2$, ranked by modified objective." Acknowledged drawback: at large $\mu$, a feature with high $q_j$ but no predictive evidence may be ranked highly by CP3 but its swap fails the raw-loss gap check, so it never enters the pool. The bonus is effectively muted for pool *membership* at large $\mu$, even though it still determines pool *ordering*. Alternatives considered:
- Absolute gap rather than relative: avoids the sign issue but loses scale invariance.
- Gap on $|L_{\text{ref}}|$: rescues sign at the cost of meaning.
- Gap on $L + \lambda_2 \|\mathbf{w}\|^2$ explicitly: current choice. Cleanest.

**Implementation pattern.** Store the modified-objective values in `sparseDiversePool_loss`; reconstruct raw loss at the gap-check site:

```python
loss_ref_raw = sparseDiversePool_loss[-1] + self.mu * self.freq[nonzero_indices].sum()
loss_swap_raw = (
    compute_logisticLoss_from_ExpyXB(sparseDiversePool_ExpyXB[sparseDiversePool_index])
    + self.lambda2 * (betas_no_old_j_squareSum + sparseDiversePool_betas[sparseDiversePool_index, new_j] ** 2)
)
if (loss_swap_raw - loss_ref_raw) / loss_ref_raw < gap_tolerance:
    # ... accept swap
```

### 2.7 CP6: parameter threading

Two new constructor parameters, `mu: float = 0.0` (math: $\mu$) and `freq: np.ndarray = None` (math: $\mathbf{q}$), threaded from `wrapper.py` and `fasterrisk.py` down to `sparseLogRegModel` and `sparseDiversePoolLogRegModel`.

**Validation in each model constructor:**
```python
self.mu = float(mu)
self.freq = np.asarray(freq) if freq is not None else np.zeros(X.shape[1])
assert self.freq.shape == (X.shape[1],), \
    f"freq must have shape ({X.shape[1]},), got {self.freq.shape}"
assert self.mu >= 0, "mu must be nonnegative"
```

Default `mu=0.0, freq=zeros(p)` makes the modification a no-op. With either default, CP1-CP4 reduce to vanilla expressions; numerical equivalence is the backward-compatibility test.

### 2.8 What does NOT need modification

| Component | Why no modification |
|---|---|
| `base_model.py`: `finetune_on_current_support` | Optimizes coefficients on a fixed support. Bonus is constant in the coefficients. |
| `rounding.py`: `AuxiliaryLossRounding` (Alg. 6) | Rounds magnitudes; never alters support. |
| `fasterrisk.py`: `StarRaySearch` (Alg. 3) | Multiplier $m$ search scales coefficients uniformly; support invariant. |
| Group-sparsity subclasses | Only override `getAvailableIndices_*`; our bonus is added *after* this filter. |
| `utils.py` | We don't redefine helpers; bonus is added at call sites. |

The integer-rounding bound transfers exactly because the bonus is invariant under support-preserving operations (§3 below).

## 3. Integer-rounding bound transfer

Liu et al. 2022 (Thm. 3.1): for the continuous sparse solution $\mathbf{w}$ and the integer-rounded $\mathbf{w}^+$,

$$
L(\mathbf{w}^+, w_0^+, \mathcal{D}) - L(\mathbf{w}, w_0, \mathcal{D}) \;\leq\; \sqrt{\, n \sum_{i=1}^n \sum_{j=1}^p \ell_i^2 \, x_{ij}^2 \, u_j (1 - u_j) \,},
$$

with $u_j = w_j - \lfloor w_j \rfloor$ and $\ell_i = 1/(1 + \exp(y_i \mathbf{x}_i^\top \boldsymbol{\gamma}_i))$ the local Lipschitz constant ($\gamma_{ij} = \lfloor w_j \rfloor$ if $y_i x_{ij} > 0$, else $\lceil w_j \rceil$).

For the modified objective:

$$
\begin{aligned}
L_{\text{mod}}(\mathbf{w}^+, w_0^+) - L_{\text{mod}}(\mathbf{w}, w_0)
&= \big[\, L(\mathbf{w}^+, w_0^+, \mathcal{D}) - L(\mathbf{w}, w_0, \mathcal{D}) \,\big] \\
&\quad + \lambda_2 \big[\, \|\mathbf{w}^+\|^2 - \|\mathbf{w}\|^2 \,\big] \\
&\quad - \mu \sum_{j=1}^p q_j \big[\, \mathbb{1}[w_j^+ \neq 0] - \mathbb{1}[w_j \neq 0] \,\big].
\end{aligned}
$$

Third bracket: identically zero (support invariant under rounding). Second bracket: small magnitude-change term vanilla FasterRisk also incurs, default $\lambda_2 = 10^{-8}$ negligible. First bracket: bounded by Thm. 3.1.

The bonus is *structurally* invariant under rounding, not just bounded under rounding. Caveat: support preservation assumes the StarRaySearch multiplier $m$ is large enough that no nonzero $|m \cdot w_j|$ rounds to zero. The bonus can drive features into the support with small $|w_j|$ (the magnitude-vs-support tension, §5.3); empirically StarRaySearch picks $m$ large enough on the regimes Liu et al. tested. Not a theoretical guarantee at extreme $\mu$.

## 4. Sanity checks

Run after implementation, before any new-source or benchmark work.

1. **$\mu = 0$ numerical equivalence.** Run existing FasterRisk tests with modified code. Outputs must match vanilla within numerical tolerance. Most important check.
2. **$\mathbf{q} = \mathbf{0}$ numerical equivalence.** Pass $\mu > 0$ but `freq = zeros(p)`. Outputs must match vanilla.
3. **$\mu \to \infty$ picks top-$k$ by $\mathbf{q}$.** Controlled instance: small synthetic where loss is approximately constant across supports (random labels, balanced features). Set $\mu = 10^6$; verify selected support equals top-$k$ features by $\mathbf{q}$.
4. **Monotonicity probe.** Sweep $\mu \in \{0, 0.01 \cdot s, 0.1 \cdot s, s, 10 s, \infty\}$ with $s = \mathrm{median}_j |\nabla_j L|$ on a synthetic with planted causal support. Plot: fraction of planted causal features selected vs. $\mu$. Expected: monotonic increase (modulo beam heuristics).
5. **Rashomon pool sanity.** At $\mu = 0$, pool $\mathcal{P}$ equals vanilla pool. At $\mu > 0$, pool members' raw losses are within `gap_tolerance` of the optimum.

## 5. Numerical and structural subtleties

### 5.1 The $\mu \to \infty$ limit is not literally "top-$k$ by $\mathbf{q}$"

`forbidden_support` (sparseBeamSearch.py line 77) deduplicates supports across the beam. As parents converge toward $\mathbf{q}$-top supports, dedup blocks duplicate proposals; the beam may end with fewer than `parent_size` surviving members. The empirical behavior is "beam concentrates on the highest-$\mathbf{q}$ supports"; the formal statement is "in the absence of beam heuristics, $k$-optimum is top-$k$ by $\mathbf{q}$."

### 5.2 Scale of $\mu$ is data-dependent

$|\nabla_j L|$ depends on $n$ and feature scale; $q_j \in [0, 1]$ does not. The sum $|\nabla L| + \mu \cdot \mathbf{q}$ in CP1 mixes scales. Practical convention: report and sweep $\mu$ in units of $\mathrm{median}_j |\nabla_j L|$ at $\mathbf{w} = \mathbf{0}$. Sweep $\{0, 0.01, 0.1, 1, 10, \infty\} \cdot \mathrm{median}|\nabla L|$, not raw values.

### 5.3 Magnitude vs support tension at large $\mu$

When the bonus heavily favors a feature, the OMP inner loop (Eq. 7 in Liu et al.) finds its optimal coefficient even if small. Low-utility features can sit in the support with near-zero $|w_j|$, kept by the bonus. AuxiliaryLossRounding combined with a small StarRaySearch multiplier $m$ may then eliminate them, undoing the bonus. Intended behavior under "causal override" framing, but caps the useful range of $\mu$ on small-effect features. Characterize empirically in stage 4.

### 5.4 Feature indexing

- $X$ is $n \times p$. `non_support`, `support` arrays in `sparseBeamSearch.py` and `sparseDiversePool.py` index the $p$ feature columns ($j = 1, \ldots, p$).
- Intercept $w_0$ stored separately as `self.beta0`; not indexed in `non_support` / `support`.
- `freq` must be length $p$, indexed consistently with $X$ columns.
- FasterRisk standardizes features (`X_norm`, `scaled_feature_indices`) but does not permute them. Verify once by reading `base_model.py` before committing.

### 5.5 $L_2$ ridge handling is asymmetric (vanilla quirk preserved)

`sparseBeamSearch.py` stores raw $L$ in `loss_arr_child` (no $\lambda_2 \|\mathbf{w}\|^2$). `sparseDiversePool.py` stores $L + \lambda_2 \|\mathbf{w}\|^2$. Vanilla FasterRisk's choice, not ours. Preserved so $\mu = 0$ matches vanilla bit-by-bit on the standard arithmetic path.

## 6. Pre-flight reads (do these before writing code, ~30 min)

1. **`base_model.py` end-to-end.** Confirm no selection-critical paths outside the two files being modified. Low probability of a surprise, worth verifying.
2. **Feature ordering.** Confirm features are not permuted by standardization (i.e., `self.betas[j]` corresponds to `X[:, j]`). If permuted, threading `freq` requires the same permutation.
3. **`forbidden_support` dedup.** Uses string keys of support sets. Confirm it does not depend on loss values (should be fine: runs before loss computation, but verify).

## 7. Expected effort

| Step | Estimate |
|---|---|
| Pre-flight reads | 30 min |
| Implement CP1, CP2 | 1 hour |
| Implement CP3, CP4, CP5 | 2 hours |
| Implement CP6 (threading) | 1 hour |
| Sanity check 1 ($\mu = 0$ equivalence) | 1 hour |
| Debug if check 1 fails | 0-4 hours |
| Sanity checks 2-5 | 2 hours |
| Buffer for numerical-tolerance surprises | 6-8 hours |
| **Total** | **~1 week realistic, 2-3 days optimistic** |

The original "1-2 days" estimate ignored that numerical-equivalence-at-$\mu=0$ debugging can absorb a day on its own. Floating-point reordering from the added arithmetic in CP1/CP2/CP4 is the likely failure mode.

The choice is not a smoothness preference. It is the unique link making the log-prior linear in $q_j$.

Start with Bernoulli on $z_j = \mathbb{1}[w_j \neq 0]$:

$$\log p(z_j) = z_j \log \pi_j + (1 - z_j) \log(1 - \pi_j) = z_j \cdot \mathrm{logit}(\pi_j) + \log(1 - \pi_j).$$

The first term is the only one that depends on $z_j$; the second is a $j$-only constant in the MAP. So the support-dependent part of the log-prior is

$$\sum_j z_j \cdot \mathrm{logit}(\pi_j).$$

We want this to equal $\mu \sum_j q_j z_j$ (linear-in-$\mathbf{q}$ bonus). Matching coefficients:

$$\mathrm{logit}(\pi_j) = \mu q_j \iff \pi_j = \sigma(\mu q_j).$$

This is the only $\pi_j$ that delivers a linear-in-$q$ MAP bonus through the Bernoulli inclusion prior. Any other link (e.g., $\pi_j = q_j$ directly, or $\pi_j = q_j^\mu$) produces a non-linear log-prior with no closed-form decomposability across features (i.e., destroys the CP1/CP3 per-feature ranking). So calling $\sigma(\mu q_j)$ "natural" understates it; it's forced by two design choices made earlier: Bernoulli prior + linear-in-$q$ bonus.



Vanilla FasterRisk reasoning. At parent $(\mathbf{w}{\text{parent}}, w_0)$, consider adding feature $j \notin \mathrm{supp}(\mathbf{w}{\text{parent}})$ with coefficient $\delta$:

$$L(\mathbf{w}{\text{parent}} + \delta \mathbf{e}j) - L(\mathbf{w}{\text{parent}}) = \delta \cdot \nabla_j L + \tfrac{1}{2} \delta^2 H{jj} + O(\delta^3).$$

Minimizing over $\delta$ gives optimal step $\delta^* \approx -\nabla_j L / H_{jj}$ and loss reduction $\approx (\nabla_j L)^2 / (2 H_{jj})$. After standardization $H_{jj}$ is roughly constant across $j$, so the loss reduction is monotonic in $|\nabla_j L|^2$, equivalently in $|\nabla_j L|$. That's the vanilla rank: top-child_size by $|\nabla_j L|$.

With the bonus. $L_{\text{mod}}$ adds the support-indicator term $-\mu q_j z_j$. When $j$ moves from $z_j = 0$ to $z_j = 1$, the bonus contributes a discrete drop of $-\mu q_j$, independent of the coefficient magnitude $\delta$. So:

$$L_{\text{mod}}(\mathbf{w}{\text{parent}} + \delta \mathbf{e}j) - L{\text{mod}}(\mathbf{w}{\text{parent}}) = \underbrace{\delta \nabla_j L + \tfrac{1}{2}\delta^2 H_{jj}}{\text{loss part}} ;-; \underbrace{\mu q_j}{\text{support flip}}.$$

At optimal $\delta^*$ the loss part is approximately $-|\nabla_j L|^2 / (2 H_{jj})$. The total optimal-step reduction is then:

$$|\Delta L_{\text{mod}}^*| \approx \frac{|\nabla_j L|^2}{2 H_{jj}} + \mu q_j.$$

Ranking by this quantity would be strictly correct under the quadratic-Taylor + standardized-Hessian assumption. The code uses the simpler proxy

$$\text{score}_j = |\nabla_j L| + \mu q_j,$$

which is rank-equivalent to $|\nabla_j L|^2 + 2 H_{jj} \mu q_j$ only under additional approximations. The simpler proxy is what vanilla FasterRisk already uses (ranking by $|\nabla_j L|$ rather than $|\nabla_j L|^2$); CP1 just adds the discrete-flip cost on the same scale.